Article 7: Mobility & Transportation Analytics on Databricks
Real-Time Fleet Routing, Traffic Analysis & Geo-Temporal Insights Using Grid Indexing

Mobility data is exploding — ride-sharing, delivery fleets, smart cities, EVs, and autonomous vehicles all continuously emit GPS pings, sensor logs, and trip metadata. The business value?

✅ Optimized fleet routing
✅ Traffic hotspot detection
✅ ETA and fuel cost forecasting
✅ Congestion-aware pricing
✅ Real-time geo-fencing alerts

With Databricks Lakehouse and spatial techniques like grid indexing, companies can scale mobility analytics from raw events → real-time intelligence without needing dedicated GIS systems.

In this article, we’ll build a complete spatial-temporal pipeline to:

Ingest vehicle GPS streams

Add grid-based spatial index

Detect congestion zones via time-windowed aggregation

Join with road network reference data

Power fleet dashboards & ML models

🗺️ Architecture Overview
[ Vehicle GPS Stream ]
     |
     v
BRONZE (raw events, Delta streaming)
     |
     v
SILVER (geo_point + grid_id + speed calc)
     |
     v
GOLD (congestion windows, route heatmaps, bottleneck alerts)
     |
     v
🖥️ BI/dashboards, Auto-routing, Mobility Pricing Engines

⚙️ 1. Simulate GPS Data + Ingest to Bronze (Streaming)

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS geo_demo;
USE CATALOG geo_demo;
CREATE SCHEMA IF NOT EXISTS mobility_bronze;
USE SCHEMA mobility_bronze;

CREATE SCHEMA IF NOT EXISTS mobility_silver;
USE SCHEMA mobility_silver;

CREATE SCHEMA IF NOT EXISTS mobility_gold;
USE SCHEMA mobility_gold;

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
import random, uuid, time

schema = StructType([
    StructField("event_id", StringType()),
    StructField("vehicle_id", StringType()),
    StructField("lat", DoubleType()),
    StructField("lon", DoubleType()),
    StructField("speed_kmh", DoubleType()),
    StructField("event_unix", DoubleType())
])

# generate ~500 random events near SF
def generate_events(n=500):
    base_lat, base_lon = 37.7749, -122.4194
    ts = int(time.time())
    return [
        (
            str(uuid.uuid4()),
            f"veh_{i % 15}",
            base_lat + random.random() * 0.03,
            base_lon + random.random() * 0.03,
            random.uniform(0, 70),
            ts - random.randint(0, 600)
        )
        for i in range(n)
    ]

df = spark.createDataFrame(generate_events(), schema) \
    .withColumn("event_time", expr("timestamp_millis(CAST(event_unix * 1000 AS BIGINT))")) \
    .withColumn("ingest_ts", current_timestamp())

df.write.format("delta").mode("overwrite").saveAsTable("mobility_bronze.events")


This simulates telemetry events and prepares them for spatial processing.

### 2. Silver Layer – Spatial + Temporal Enrichment
Add spatial geometry, grid index, and calculate speed-based features.

In [0]:
%sql
CREATE OR REPLACE TABLE mobility_silver.events AS
SELECT
  event_id,
  vehicle_id,
  event_time,
  lat,
  lon,
  speed_kmh,
  ST_POINT(lon, lat) AS geo_point,
  CONCAT(FLOOR(lat * 100), '_', FLOOR(lon * 100)) AS grid_id,
  CAST(event_time AS DATE) AS day,
  ingest_ts
FROM mobility_bronze.events
WHERE lat BETWEEN -90 AND 90 AND lon BETWEEN -180 AND 180;


### 3. Gold Layer – Geo-Temporal Congestion Detection
Step A: Aggregate by Grid + Time Window

In [0]:
%sql
CREATE OR REPLACE TABLE mobility_gold.grid_5m AS
SELECT
  grid_id,
  WINDOW(event_time, '5 minutes') AS t,
  COUNT(*) AS total_events,
  AVG(speed_kmh) AS avg_speed,
  COUNT(DISTINCT vehicle_id) AS vehicles_active
FROM mobility_silver.events
GROUP BY grid_id, WINDOW(event_time, '5 minutes');


Step B: Identify Congestion Zones (slow speed + high volume)

In [0]:
%sql
CREATE OR REPLACE TABLE mobility_gold.congestion_zones AS
SELECT
  grid_id,
  t.start AS start_time,
  vehicles_active,
  avg_speed
FROM mobility_gold.grid_5m
WHERE avg_speed < 15 AND vehicles_active > 8
ORDER BY start_time DESC;


Step C: Create Route Heatmap for Fleet Managers

In [0]:
%sql
CREATE OR REPLACE TABLE mobility_gold.route_density AS
SELECT
  grid_id,
  day,
  COUNT(*) AS total_movements,
  APPROX_PERCENTILE(speed_kmh, 0.5) AS median_speed
FROM mobility_silver.events
GROUP BY grid_id, day;


Business Impact

With the Gold tables, fleet operators can:

Use Case	Action Triggered
Live congestion zone alerts	Reroute drivers automatically
Route heatmaps	Optimize dispatch regions or avoid red grids
Median speed by tile	Support pricing models and fuel optimization
Vehicle clustering	Find idle & underutilized assets
Cross-join with roads	Identify geographic areas with structural issues

### Join with Road Network Dataset

In [0]:
%sql
SELECT
  r.road_name,
  c.grid_id,
  c.avg_speed,
  c.vehicles_active
FROM mobility_gold.grid_5m c
JOIN mobility_silver.road_segments r
  ON c.grid_id = r.grid_id
WHERE c.avg_speed < 10;


This can drive public transportation optimization, traffic signal adjustments, and smart-city insights.

Scaling & Performance Patterns
Technique	Why It Helps
Use grid_id before ST_DISTANCE	Avoids geometry heavy scans
ZORDER on grid_id and event_time	Fast predicate pruning
Photon on SQL warehouse	Vectorized geo execution
Partition by day in Silver/Gold	Efficient dashboard filtering

Add ML for Arrival Time Predictions

Once your Silver and Gold layers are ready, build ETA models based on:

Grid-level median speeds

Historical traffic patterns

Time of day / day of week

Weather joined by spatial region

Use MLflow to track experiments:

In [0]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.feature import VectorAssembler
import mlflow

df = spark.table("mobility_gold.grid_5m") \
    .withColumn("hour", hour("t.start")) \
    .withColumn("label", expr("avg_speed"))

feat = df.select("vehicles_active", "hour", "label")

vec = VectorAssembler(inputCols=["vehicles_active", "hour"], outputCol="features")
rf = RandomForestRegressor(featuresCol="features", labelCol="label")

with mlflow.start_run():
    model = rf.fit(vec.transform(feat))
    mlflow.spark.log_model(model, "speed_prediction_model")


Summary Deliverables
✅ Real-time GPS ingest → Bronze
✅ Grid-indexed spatial enrichment → Silver
✅ Congestion grid + fleet density → Gold
✅ Supports fleet routing, dispatching, pricing & compliance
✅ Built using only Databricks Serverless — no H3 needed

Would you like to extend this with tolling analytics, EV route energy modeling, or a Databricks SQL real-time dashboard?